# FastAPI para Machine Learning

## Introdução

### Iniciando um Projeto

Neste guia estaremos utilizando o Uvicorn e Virtual Environments para gerenciar as dependências do projeto. Para instalar o framework do Fast API, devemos executar o comando abaixo.

In [ ]:
uv add "fastapi[standard]"

Abaixo, um exemplo de API mínima. Primeiro, importamos a classe FastAPI da biblioteca fastapi e instanciamos ela, chamando a variável preferencialmente de app. Para definir uma rota, utilizamos alguns decorators de app de acordo com o método de requisição que queremos utilizar. Os decorators devem ser aplicados em funções e recebem como primeiro argumento o endpoint da rota. Esta rota de exemplo, quando acessada, retornará um JSON com uma chave "message" e valor "Aprendendo Fast API".

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def root():
    return { "message": "Aprendendo Fast API" }    

### Estrutura do Projeto

Atualmente, é assim que está a estrutura de pastas do projeto. Neste guia, estaremos utilizando como base um projeto completo de Machine Learning para predição de churn no dataset público do bigquery de eventos do GA4. Em casos como esse, em que precisamos disponibilizar um modelo através do Fast API, o ideal é criar uma pasta app dentro de src. Nesta pasta app, teremos todo o código da API, enquanto na pasta vizinha (ga4_churn_predictor) teremos todo o código de Machine Learning. Observe abaixo a estrutura de pastas com descrição.

```text
├── models
│   └── churn_model.joblib
├── notebooks
│   └── eda_01.ipynb
├── sql
│   └── ga4_event_query.sql
├── src
│   ├── app                        # Isola todo o código da API
│   │   ├── routers                # Contém a definição de todas as rotas
│   │   │   └── __init__.py
│   │   ├── schemas                # Abriga todos os schemas com Pydantic
│   │   │   └── __init__.py
│   │   ├── services               # Encapsula toda a lógica de negócio
│   │   │   └── ml_service.py
│   │   ├── config.py              # Guarda todas as variáveis de ambiente e constantes
│   │   ├── database.py            # Gerencia a sessão do banco de dados, se necessário
│   │   ├── security.py            # Fornece as dependências necessárias para a segurança da API
│   │   └── main.py                # O ponto de entrada, onde a API começa a rodar
│   └── ga4_churn_predictor
│       ├── __init__.py
│       ├── config.py
│       ├── data.py
│       ├── evaluate.py
│       ├── pipeline.py
│       ├── predict.py
│       └── train.py
├── pyproject.toml
├── README.md
└── uv.lock
```

### Rodando a API

Abaixo, está o comando que executa o código para disponibilizar a API no localhost. Para isso, utilizamos uv run uvicorn. Depois desse prefixo, devemos especificar onde exatamente está a variável que instanciou a classe FastAPI. Como estamos na raíz do projeto, a variável está dentro na pasta src/app em um arquivo chamado main.py e com o nome "app". Primeiro separamos cada pasta e arquivo com pontos e quando estivermos no arquivo em que a API é iniciada, usamos dois pontos para indicar o nome da variável. Por fim, a flag reload fará com que ao editar o código, o processo reinicie. Dessa forma, a API será executada em hot reload. Ao executar o comando, o uvicorn indicará em que porta a API está disponível. 

```shell
uv run uvicorn src.app.main:app --reload
```

### Configurando DotEnv

DotEnv é um recurso crucial que tem como propósito esconder informações secretas do projeto, mantendo a segurança. É nele que definimos variáveis de ambiente como por exemplo o endereço e senha do banco de dados, chaves de api, etc. Para utilizar esse recurso no FastAPI, antes de tudo, precisamos instalar a biblioteca python-dotenv. Que nos permitirá configurar as variáveis de ambiente. Para instalar essa biblioteca, rodamos o comando abaixo.

```shell
uv add python-dotenv
```

Como ficou evidente na descrição da estrutura de pastas, o lugar onde gerenciamos as variáveis de ambiente é o arquivo config.py. Logo, é exatamente nele que realizaremos a configuração da biblioteca. Para isso, precisamos de duas outras coisas primeiro: criar um arquivo .env; localizar o arquivo no código python. O arquivo .env é exatamente onde escrevemos as variáveis secretas do projeto no formato CHAVE=VALOR.

Para determinar o caminho absoluto da raíz do projeto, utilizamos o pathlib. A variável que representa a raíz do projeto no exemplo abaixo é ROOT_DIR. Já o caminho até o .env é exatamente a junção do caminho até a raíz do projeto com "/.env". Em seguida, chamamos a função load_dotenv da biblioteca dotenv. E está pronto! Agora, já podemos acessar as variáveis do arquivo .env no código python.

In [ ]:
from pathlib import Path
from ga4_churn_predictor.config import ROOT_DIR
from dotenv import load_dotenv

ROOT_DIR = Path(__file__).resolve().parents[2]
ENV_PATH = ROOT_DIR / ".env"

load_dotenv(ENV_PATH)

IMPORTANTE: O arquivo .env JAMAIS deve ser publicado no Github ou em qualquer outro local público. Mas é uma boa prática publicar um .env.example que contenha apenas as chaves para que as pessoas saibam de que variáveis o sistema precisa.

### Utilizando DotEnv e Documentação Automática

Uma das vantagens de utilizar FastAPI é o recurso de documentação automática. Com base em decorators e annotations presentes no código, será gerada automaticamente uma documentação em swagger para a API no endpoint /docs. Supondo que a API tenha sido disponibilizada no localhost, porta 8000, o endereço de acesso à documentação seria: http://localhost:8000/docs.

Algumas das informações da documentação automática que podemos especificar são: versão da API; título da API; descrição da API. Podemos guardar essas informações dentro do .env que acabamos de configurar. Para fornecer essas informações, demos conjuntos de CHAVE=VALOR para elas no .env conforme representado abaixo.

```dotenv
API_TITLE="GA4 Churn Predictor API"
API_VERSION="1.0.0"
API_DESCRIPTION="API de Predição de Churn com Machine Learning"
```

Depois, no arquivo config.py, sempre abaixo do load_dotenv, referenciamos as variáveis de ambiente que definimos no .env. Afinal, se fizéssemos isso antes da chamada do load_dotenv, não conseguiríamos utilizar as variáveis, já que o dotenv não teria sido configurado ainda. Para acessar os valores que estão no .env, utilizamos a biblioteca nativa "os" que fornece diversas funções do sistema operacional. Utilizando o método getenv dessa biblioteca, podemos recuperar os valores através da chave. Como primeiro argumento para o método, temos de passar exatamente a chave como está escrita no .env. Como segundo argumento, podemos passar um valor padrão para caso aquela variável que tentamos recuperar não exista no arquivo.

In [ ]:
import os
from pathlib import Path
from ga4_churn_predictor.config import ROOT_DIR
from dotenv import load_dotenv

ROOT_DIR = Path(__file__).resolve().parents[2]
ENV_PATH = ROOT_DIR / ".env"

load_dotenv(ENV_PATH)

API_VERSION=os.getenv("API_VERSION", "1.0.0")
API_TITLE=os.getenv("API_TITLE", "GA4 Churn Predictor API")
API_DESCRIPTION=os.getenv("API_DESCRIPTION", "API de Predição de Churn com Machine Learning")

Por fim, podemos passar os valores coletados do .env para a classe FastAPI como parâmetros title, version e description. Dessa forma, quando abrirmos a documentação no navegador, o título, a descrição e a versão da API exibidas serão como definimos no .env.

In [ ]:
from app.config import API_TITLE, API_VERSION, API_DESCRIPTION
from fastapi import FastAPI

app = FastAPI(
    title=API_TITLE,
    version=API_VERSION,
    description=API_DESCRIPTION,
)

@app.get("/")
def root():
    return { "message": "Aprendendo Fast API" }    

## Validação com Pydantic

No FastAPI, utilizamos o Pydantic para definir a garantia e integridade de entrada e o contrato de resposta. Ou seja, define muito bem o que devemos receber e o que devemos retornar.

Respeitando a estrutura do projeto, nossos esquemas de validação devem estar sempre dentro da pasta schemas. Portando, dentro dela, criamos o arquivo prediction.py. Nele, teremos duas classes. Uma para validar o input e outra para validar o output.

Para definir um esquema de validação, precisamos utilizar uma classe que extenda a classe BaseModel do Pydantic. Dentro dela, devemos especificar uma propriedade para cada campo que tem de ser recebido ou retornado. Para cada campo, devemos também especificar o tipo da propriedade e finalmente atribuir ao campo um objeto da classe Field do Pydantic. É nesse objeto e na classe BaseModel que está a mágica.

O Field pode receber ou não o primeiro argumento. Podendo este ser ```...```, ```None``` ou um valor padrão para o campo. No Pydantic V2, quando queremos que um campo seja obrigatório, podemos utilizar o ```...``` ou simplesmente não definir o primeiro argumento. 

Observe o exemplo abaixo. Todos os campos que definimos são obrigatórios. Alguns do tipo int e alguns do tipo float. O Field pode receber outros argumentos como ge, gt, le e lt. O ge significa "greater or equal" (maior ou igual). O gt significa "greater than" (maior que). O le significa "lower or equal" (menor ou igual) e o lt significa "lower than" (menor que). Estes argumentos são utilizados para campos numéricos, definindo os valores mínimos e máximos que o campo pode receber. Além disso, também podemos definir uma descrição para cada campo utilizando o argumento description.


In [ ]:
from pydantic import Field
from pydantic import BaseModel


class ChurnInput(BaseModel):
    total_pedidos: int = Field(..., gt=0, description="...")
    receita_total_em_usd: float = Field(..., gt=0, description="...")
    ticket_medio_em_usd: float = Field(..., gt=0, description="...")
    recencia_dias: int = Field(..., ge=0, description="...")
    dias_desde_primeiro_pedido: int = Field(..., ge=0, description="...")
    tempo_relacionamento_dias: int = Field(..., ge=0, description="...")
    total_itens_comprados: int = Field(..., ge=0, description="...")
    total_interacoes_quinzena_1: int = Field(..., ge=0, description="...")
    total_interacoes_quinzena_2: int = Field(..., ge=0, description="...")
    total_interacoes_quinzena_3: int = Field(..., ge=0, description="...")
    total_interacoes_quinzena_4: int = Field(..., ge=0, description="...")


class ChurnOutput(BaseModel):
    predicao_classe: int = Field(..., ge=0, le=1, description="...")
    predicao_probabilidade: float = Field(..., ge=0, le=1, description="...")
    predicao_status: str = Field(..., description="...")

Para fins didáticos. Eis o código completo dos esquemas.

In [ ]:
from pydantic import Field
from pydantic import BaseModel


class ChurnInput(BaseModel):
    total_pedidos: int = Field(..., gt=0, description="Total de pedidos")
    receita_total_em_usd: float = Field(
        ..., gt=0, description="Receita total em dólares"
    )
    ticket_medio_em_usd: float = Field(..., gt=0, description="Ticket médio em dólares")
    recencia_dias: int = Field(..., ge=0, description="Dias desde o último pedido")
    dias_desde_primeiro_pedido: int = Field(
        ..., ge=0, description="Dias desde o primeiro pedido"
    )
    tempo_relacionamento_dias: int = Field(
        ..., ge=0, description="Tempo total de relacionamento em dias"
    )
    total_itens_comprados: int = Field(
        ..., ge=0, description="Total de itens comprados"
    )
    total_interacoes_quinzena_1: int = Field(
        ..., ge=0, description="Total de interações na primeira quinzena"
    )
    total_interacoes_quinzena_2: int = Field(
        ..., ge=0, description="Total de interações na segunda quinzena"
    )
    total_interacoes_quinzena_3: int = Field(
        ..., ge=0, description="Total de interações na terceira quinzena"
    )
    total_interacoes_quinzena_4: int = Field(
        ..., ge=0, description="Total de interações na quarta quinzena"
    )


class ChurnOutput(BaseModel):
    predicao_classe: int = Field(..., ge=0, le=1, description="Predição de churn")
    predicao_probabilidade: float = Field(
        ..., ge=0, le=1, description="Probabilidade de churn"
    )
    predicao_status: str = Field(..., description="Status da predição")

### Validação do .env com Pydantic

Antes de tudo, para implementar validação do .env com Pydantic, precisamos instalar a biblioteca abaixo. Ela nos fornecerá utilitários que nos ajudarão a cumprir essa tarefa.

```shell
uv add pydantic-settings
```

Se lembra do arquivo .env para configuração das variáveis de ambiente? Existe uma forma mais recomendada para gerenciar essas variáveis. Essa abordagem utiliza o Pydantic para validar as variáveis. Primeiro, assim como antes, definimos o caminho até o .env. Feito isso, definimos uma classe que extende a classe BaseSettings da biblioteca que acabamos de instalar. Dentro dessa classe, definimos todas as variáveis que queremos validar e utilizamos a classe Field para ditar as regras. Finalmente, definimos uma propriedade model_config na classe, que instancia a classe SettingsConfigDict. Essa classe definirá configurações para a validação do .env. Ela receberá um argumento env_file que deve receber o caminho até o .env, receberá também um argumento env_file_encoding que define o método de encoding do arquivo .env. Finalmente, ela pode receber um argumento extra que define o que deverá acontecer caso o .env tenha chaves adicionais que não estão definidas no esquema ainda. O valor "ignore" fará com que essas variáveis sejam ignoradas. Então quando adicionarmos uma variável, teremos obrigatóriamente de definir regras de validação para ela antes de utilizá-la.

Terminada a definição da classe, instanciamos ela e definimos uma variável no escopo global para cada propriedade que ela tem. Assim, poderemos importar as variáveis em outros módulos.

Mas afinal, quais são as vantagens de utilizar isso? É muito simples. Se você tentar executar a API sem definir todas as variáveis de ambiente, será avisado logo. Se você tentar referenciar uma variável pelo nome errado, você também será avisado logo. Além disso, os tipos são convertidos automaticamente. Em um arquivo .env, absolutamente tudo é texto. Mas quando usamos o Pydantic e definimos que uma variável é um inteiro, ela será convertida para int automaticamente.

In [ ]:
from pathlib import Path
from pydantic import Field
from pydantic_settings import SettingsConfigDict
from pydantic_settings import BaseSettings

ROOT_DIR = Path(__file__).resolve().parents[2]
ENV_PATH = ROOT_DIR / ".env"


class Settings(BaseSettings):
    API_TITLE: str = Field(..., description="Título da API")
    API_VERSION: str = Field(..., description="Versão da API")
    API_DESCRIPTION: str = Field(..., description="Descrição da API")

    model_config = SettingsConfigDict(
        env_file=ENV_PATH,
        env_file_encoding="utf-8",
        extra="ignore"
    )


settings = Settings()

API_VERSION=settings.API_VERSION
API_TITLE=settings.API_TITLE
API_DESCRIPTION=settings.API_DESCRIPTION

## Injeção de Dependência

Para demonstrar como é feita a injeção de dependências no FastAPI, criamos um código de conexão ao banco de dados SQLite.

No código abaixo, a conexão é feita com SQLite em arquivo local. A função get_db cria um novo objeto de sessão de conexão, "empresta" ele para o contexto de execução de uma requisição e quando a requisição termina, fecha a conexão.

In [ ]:
from sqlalchemy.orm import declarative_base
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

DATABASE_URL = "sqlite:///./banco.db"

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

def get_db():
    db = SessionLocal()

    try:
        yield db
    finally: 
        db.close()

No exemplo inicial de uma rota, é assim que fica o uso. Nas definições de argumentos da função que processa a requisição, incluímos um novo argumento do tipo Session do sqlalchemy que recebe um objeto da classe Depends. Esse objeto é que realiza a injeção de dependência. Para a classe Depends, passamos como parâmetro a função get_db que providencia a dependência que há de ser injetada. Nesse caso, a sessão no banco de dados. Como o banco de dados propriamente não é o mais importante para o propósito ao qual estamos aplicando FastAPI, não entraremos em detalhe. Este é apenas um exemplo de como funciona a injeção de dependência.

In [ ]:
from sqlalchemy import text
from sqlalchemy.orm import Session
from app.database import get_db
from fastapi import Depends
from app.config import API_TITLE, API_VERSION, API_DESCRIPTION
from fastapi import FastAPI

app = FastAPI(
    title=API_TITLE,
    version=API_VERSION,
    description=API_DESCRIPTION,
)

@app.get("/")
def root(db_session: Session = Depends(get_db)):
    # Código que utiliza a conexão com o banco de dados

    return { "message": "Aprendendo Fast API" }    

## Camada de Serviço

Conforme avançarmos, ficará claro que cada coisa tem uma única e exclusiva responsabilidade. A responsabilidade da camada de serviço é encapsular toda a regra de negócio. Como nosso projeto é focado em expor o modelo que treinamos ao mundo, então o serviço mais essencial encapsula as funções de predição.

Portanto, definimos uma classe MLService. O constructor dessa classe inicializa uma propriedade model que será o modelo utilizado para as predições. Definimos um método load_model que carregará um novo modelo utilizando a função load_model definida no arquivo de predição e definirá ele como o modelo que a classe MLService deve usar. 

In [ ]:
from app.schemas.prediction import ChurnOutput
from app.schemas.prediction import ChurnInput
from ga4_churn_predictor.predict import load_model
from ga4_churn_predictor.predict import predict_single, predict_batch


class MLService:
    def __init__(self):
        self.model = None

    def load_model(self):
        self.model = load_model()

    # ...

Por fim, definimos os métodos de predição, começando com o predict_single que receberá um objeto da classe ChurnInput, que é o esquema de validação que definimos anteriormente e retornará ChurnOutput que é o esquema de validação da resposta. Dessa forma, o serviço já recebe e retorna os dados exatamente no formato que deveria. A função deverá retornar um erro caso um modelo não tenha sido carregado ainda. Depois, chama a função predict_single do arquivo de funções de predição, passando todos os parâmetros necessários e retorna um objeto do tipo ChurnOutput com os resultados da predição.

In [ ]:
# ...

class MLService:
    # ...
    
    def predict_single(self, data: ChurnInput) -> ChurnOutput:
        if self.model is None:
            raise RuntimeError("O modelo não foi carregado")

        output = predict_single(
            self.model,
            total_pedidos=data.total_pedidos,
            receita_total_em_usd=data.receita_total_em_usd,
            ticket_medio_em_usd=data.ticket_medio_em_usd,
            recencia_dias=data.recencia_dias,
            dias_desde_primeiro_pedido=data.dias_desde_primeiro_pedido,
            tempo_relacionamento_dias=data.tempo_relacionamento_dias,
            total_itens_comprados=data.total_itens_comprados,
            total_interacoes_quinzena_1=data.total_interacoes_quinzena_1,
            total_interacoes_quinzena_2=data.total_interacoes_quinzena_2,
            total_interacoes_quinzena_3=data.total_interacoes_quinzena_3,
            total_interacoes_quinzena_4=data.total_interacoes_quinzena_4,
        )

        return ChurnOutput(
            predicao_classe=output["predicao_classe"],
            predicao_probabilidade=output["predicao_probabilidade"],
            predicao_status=output["predicao_status"],
        )

Em seguida, definimos o método predict_batch para predições em lote. Como o pacote de predição espera uma lista de dicionários para processar tudo de forma vetorizada com o Pandas, convertemos a lista de ChurnInput com [item.model_dump() for item in data]. Para validar e transformar a lista de dicionários retornada em uma lista de ChurnOutput com alta performance e sem criar loops manuais, utilizamos o TypeAdapter do Pydantic.

In [ ]:
# ...
from pydantic import TypeAdapter

churn_output_list_adapter = TypeAdapter(list[ChurnOutput])

class MLService:
    # ...
    
    def predict_batch(self, data: list[ChurnInput]) -> list[ChurnOutput]:
        if self.model is None:
            raise RuntimeError("O modelo não foi carregado")

        data_as_dict_list = [item.model_dump() for item in data]

        output = predict_batch(
            self.model,
            data=data_as_dict_list
        )

        return churn_output_list_adapter.validate_python(output)

Mas por quê o serviço recebe ChurnInput e retorna ChurnOutput? E agora veja qual é a responsabilidade dos serviços no geral. Além de encapsular a regra de negócio, na arquitetura do sistema, os serviços tem como objetivo primário criar uma ponte entre HTTP e o coração do sistema. Seja ele um banco de dados, um modelo de inteligência artificial ou qualquer outro motor.

Agora, tem duas formas de utilizarmos o serviço. A simples, ideal para protótipos rápidos. E a mais robusta com injeção de dependência.

### Forma Simples

Na forma simples, instanciamos a classe MLService diretamente no final do arquivo ml_service.py. Dessa forma, ela se torna um singleton e basta importá-la onde for necessário, como nas rotas ou no evento de inicialização.

In [ ]:
# ...

ml_service = MLService()

### Forma Robusta com Injeção de Dependência

Na forma com injeção de dependência, criamos uma função para injetar o serviço nas rotas usando Depends. A principal vantagem dessa abordagem é a testabilidade: em testes unitários e de integração, podemos facilmente substituir o serviço real por um mock, sem precisar carregar o modelo na memória. A função tem de retornar uma instância de MLService em singleton. Caso contrário, uma nova instância será criada a cada vez que injetarmos o serviço.

In [ ]:
from fastapi import Depends
from app.services.ml_service import MLService, ml_service

def get_ml_service() -> MLService:
    return ml_service

# @app.get("/predict")
# def predict(service: MLService = Depends(get_ml_service)):
#     return service.predict_single(...)

## Roteamento Modular

Este é o momento em que finalmente as funcionalidades do sistema se tornarão disponíveis para uso. Reforçamos que, em uma arquitetura de sistema robusta, cada coisa tem uma única e exclusiva responsabilidade. Atualmente, o modelo não conhece os serviços e nem sabe o que é FastAPI. Já os serviços não conhecem o que é HTTP e encapsulam toda a regra de negócio. E é exatamente assim que tem de ser. 

Qual é, afinal, a tarefa das rotas? As rotas devem desempenhar uma única tarefa, que é ligar as requisições HTTP aos serviços, sem definir regra de negócio. Afinal, a regra de negócio deve ser gerenciada apenas pelos serviços.

Agora que sabemos o que uma rota deve fazer, vamos começar a implementar as rotas. Anteriormente definimos uma rota simples utilizando diretamente o objeto app. Mas, a forma mais ideal de fazer isso não é dessa forma, e sim, utilizando roteamento modular. Para definir as rotas de predição, criamos um arquivo predict.py dentro da pasta routers. Nesse arquivo, criamos uma variável router que instanciará a classe APIRouter do FastAPI. Essa classe pode receber alguns argumentos, dentre eles: prefix e tags. O argumento prefix definirá um prefixo para o roteador. Qualquer rota inclusa nele precisará incluir este prefixo no endpoint. Por exemplo, se anexarmos esse roteador diretamente em app e a API estiver disponível em ```http://localhost:8000```, então as rotas deste roteador estarão sob o endereço ```http://localhost:8000/predict```. Se lembra que dissemos que devemos utilizar algumas anotações no código a fim de gerenciar a documentação automática? O argumento tags receberá uma lista de tags que separam as rotas na página de documentação. Sem utilizar esse argumento nos roteadores, todas as rotas ficarão juntas e misturadas. Sem separação alguma.

In [ ]:
from fastapi import APIRouter

router = APIRouter(prefix="/predict", tags=["Predições"])

Agora, para definir as rotas de fato, utilizaremos decorators do router como router.post, router.get, etc. Nesse caso, receberemos requisições do tipo POST. Como primeiro argumento, os decorators recebem o nome da rota. Podem também receber um argumento response_model que deverá receber um esquema do Pydantic. Quando a função que processa a requisição recebe um objeto cujo tipo é um esquema do Pydantic, o FastAPI sabe que ele representa o corpo da requisição. Na rota ```/single```, estamos recebendo como corpo da requisição um objeto do tipo ChurnInput. Além disso, também injetamos um objeto do tipo MLService. A única coisa que a rota faz é chamar o método predict_single de MLService, passando como parâmetro o corpo da requisição e retornar o resultado. O uso de response_model adicionará na documentação um exemplo de resposta para a rota. Assim como usar um esquema do Pydantic para o corpo da requisição adicionará um exemplo de corpo de requisição.

In [ ]:
# ...

from app.schemas.prediction import ChurnOutput
from fastapi import Depends
from app.schemas.prediction import ChurnInput
from app.services.ml_service import get_ml_service
from app.services.ml_service import MLService

@router.post("/single", response_model=ChurnOutput)
def predict_single(data: ChurnInput, ml_service: MLService = Depends(get_ml_service)):
    return ml_service.predict_single(data)

Também definimos uma rota do tipo POST /batch que receberá uma lista de objetos do tipo ChurnInput pelo corpo da requisição, chamará o método predict_batch de MLService e retornará como resposta uma lista de ChurnOutput.

In [ ]:
@router.post("/batch", response_model=list[ChurnOutput])
def predict_batch(data: list[ChurnInput], ml_service: MLService = Depends(get_ml_service)):
    return ml_service.predict_batch(data)

Essas duas rotas, se o endereço da API for ```http://localhost:8000```, por exemplo, estarão disponíveis em ```http://localhost:8000/predict/single``` e ```http://localhost:8000/predict/batch```, respectivamente.

Para incluir as rotas do roteador que acabamos que configurar, basta passá-lo como parâmetro ao chamar o método include_router de app.

In [ ]:
from app.config import API_TITLE, API_VERSION, API_DESCRIPTION
from fastapi import FastAPI
from app.routers.predict import router as predict_router

app = FastAPI(
    title=API_TITLE,
    version=API_VERSION,
    description=API_DESCRIPTION,
)

app.include_router(predict_router)

## Parameterização

|Classe| O que faz |
|------------------------|---|
| Path() | Recebe parâmetros do tipo path. Ex: /clientes/1 |
| Query() | Recebe parâmetros do tipo query. Ex: /clientes?id=1 |
| Header() | Recebe parâmetros através dos headers da requisição |
| Cookie() | Recebe um cookie da requisição que tenha o mesmo nome |

### Path Params

Para utilizar parâmetros HTTP no geral, a estrutura será essencialmente a mesma. A função que lida com a requisição (nesse caso, path_params) receberá um argumento, cujo nome deve ser idêntico ao nome do path param, query param, header ou cookie, conforme o que estiver sendo utilizado. 

Em seguida, pode ser definido opcionalmente um tipo para o argumento. Por fim, o argumento da função terá de receber um objeto do tipo Path, Query, Header ou Cookie. Todas essas classes devem ser importadas do FastAPI.

In [ ]:
# ...

from fastapi import Path

@app.get("/path-params/{client_id}")
def path_params(
    client_id: int = Path(..., ge=1, description="ID do cliente")
    ):
    return {
        "client_id": client_id
    }

Caso queiramos atribuir um nome à variável que seja diferente do utilizado no nome da rota, basta usar o parâmetro alias ao instanciar a classe de validação de parâmetros. Confira o exemplo abaixo. Isso também é aplicável para as outras formas de receber parâmetros.

In [ ]:
# ...

from fastapi import Path

@app.get("/path-params/{id_cliente}")
def path_params(
    client_id: int = Path(..., ge=1, description="ID do cliente", alias="id_cliente")
    ):
    return {
        "client_id": client_id
    }

#### Exemplos de Requisição

```shell
curl -X GET "http://localhost:8000/path-params/34634"
curl -X GET "http://localhost:8000/path-params/87523"
curl -X GET "http://localhost:8000/path-params/7643"
```

### Query Params

Nos query params, mantemos a mesma linha. O nome da variável deve ser idêntico ao nome do parâmetro na URL. Observe que usamos classes utilitárias como Optional e List da biblioteca typing. A classe List define que um argumento receberá uma lista de valores de um determinado tipo, que deve estar entre colchetes. Já a classe Optional define que um argumento é opcional e de um determinado tipo, que também deve estar entre colchetes, conforme o exemplo abaixo.

In [ ]:
# ...

from typing import Optional, List
from fastapi import Query

@app.get("/query-params")
def query_params(
    search: Optional[str] = Query(min_length=3, description="Termo de busca"),
    roles: Optional[List[str]] = Query(None, enum=["user", "admin"]),
    page: int = Query(1, ge=1, description="Página da paginação"),
    per_page: int = Query(10, ge=1, le=50, description="Número de registros por página"),
    order_by: str = Query("created_at", description="Atributo para ordenação"),
    order_mode: str = Query("desc", enum=["asc", "desc"], description="Modo de ordenação"),
    ):
    return {
        "search": search,
        "roles": roles,
        "page": page,
        "per_page": per_page,
        "order_by": order_by,
        "order_mode": order_mode
    }

ATENÇÃO: O parâmetro enum não valida o valor recebido, apenas adiciona uma anotação à documentação automática, informando os valores que o campo deve receber. Ou seja, se o valor recebido não estiver incluso na lista de opções do enum, ele ainda será aceito. 

Para validar de fato o valor recebido dentre uma lista de opções, devemos utilizar o tipo Literal da biblioteca typing. Entre colchetes, listamos todas as opções de valores que a variável pode ter. Dessa forma, a validação ocorrerá de fato.

In [ ]:
# ...

from typing import Literal

@app.get("/query-params")
def query_params(
    search: Optional[str] = Query(min_length=3, description="Termo de busca"),
    roles: List[Literal["user", "admin"]] = Query(enum=["user", "admin"]),
    page: int = Query(1, ge=1, description="Página da paginação", alias="page_number"),
    per_page: int = Query(10, ge=1, le=50, description="Número de registros por página"),
    order_by: str = Query("created_at", description="Atributo para ordenação"),
    order_mode: Literal["asc", "desc"] = Query("desc", enum=["asc", "desc"], description="Modo de ordenação"),
    ):
    return {
        "search": search,
        "roles": roles,
        "page": page,
        "per_page": per_page,
        "order_by": order_by,
        "order_mode": order_mode
    }

#### Exemplos de Requisição

```shell
curl -X GET "http://localhost:8000/query-params?search=pesquisa"
curl -X GET "http://localhost:8000/query-params?search=aprendendo%20fastapi"
curl -X GET "http://localhost:8000/query-params?roles=user&roles=admin"
curl -X GET "http://localhost:8000/query-params?page=2&per_page=10"
curl -X GET "http://localhost:8000/query-params?order_by=name&order_mode=asc"
```

### Headers

Agora que já estamos no terceiro tipo de parâmetro, uma coisa fica muito clara: as classes de validação dos parâmetros são incrivelmente similares ao Field do Pydantic. Isso acontece por um motivo muito simples. Todas essas classes herdam de classes de validação do Pydantic. Ou seja, a maioria do que aprendemos ao utilizar o Pydantic é aplicável para os parâmetros. Sendo assim, o uso de ```...``` ou a falta de um valor padrão definem o parâmetro como obrigatório, mesmo quando usamos a classe Optional de typing.

In [ ]:
# ...

from typing import Optional
from fastapi import Header

@app.get("/header-params")
def header_params(
    api_key: Optional[str] = Header(..., description="Chave de acesso da API", alias="api-key"),
    user_agent: Optional[str] = Header(None),
    ):
    return {
        "api_key": api_key,
        "user_agent": user_agent
    }

#### Exemplo de Requisição

```shell
curl -X GET "http://localhost:8000/header-params" -H "api-key: chave secreta"
```

### Cookies

Além de tudo isso, as classes de validação de parâmetros, como o nome já diz, não servem apenas para tipar ou indicar características sobre os parâmetros. Mas também para validar de fato os valores recebidos.

In [ ]:
# ...

from typing import Optional
from fastapi import Cookie

@app.get("/cookie-params")
def cookie_params(
    session_id: Optional[str] = Cookie(None)
    ):
    return {
        "session_id": session_id
    }

#### Exemplo de Requisição

```shell
curl -X GET "http://localhost:8000/cookie-params" -b "session_id=sessao_8"
```

### Tabela de Parâmetros de Validação

| Nome do Parâmetro | O que Define |
|------------------------|---|
| description | Descrição do parâmetro |
| alias | O nome real do parâmetro da requisição |
| deprecated | Se o parâmetro se tornou obsoleto |
| ge | Valor que o parâmetro deve ser maior ou igual |
| gt | Valor que o parâmetro deve ser maior |
| le | Valor que o parâmetro deve ser menor ou igual |
| lt | Valor que o parâmetro deve ser menor |
| min_length | Tamanho mínimo de uma string |
| max_length | Tamanho máximo de uma string |
| enum | Quais valores o parâmetro pode receber |
| pattern | Expressão regular para validação do valor do parâmetro |

## Segurança e Autenticação por API Key

Agora que já temos o modelo treinado e as rotas de predição já estão expostas, precisamos pensar em segurança. No geral, APIs que têm como propósito expor um modelo de Machine Learning como o nosso, são consumidas não por usuários finais, e sim por outros sistemas. Portanto, a autenticação ocorre de máquina para máquina e não é apropriado realizá-la através de credenciais de usuário. A forma mais ideal e profissional é autenticar por uma chave de API.

**Em resumo, o método consiste em**:

1. Definir uma ou mais chaves de API autorizadas
2. Receber um header com a chave de API nas requisições autenticadas
3. Validar se a chave recebida está entre as chaves autorizadas
4. Caso esteja, liberar o acesso
5. Caso contrário, retornar um erro

Há duas formas de implementar a autenticação por API Key. Primeiro apresentaremos a forma simples para facilitar o entendimento e em seguida apresentaremos a forma robusta e mais recomendada.

### Método Simples

Neste primeiro método, antes de tudo precisamos criar uma função que valide a chave de API recebida. Para isso, a função deve receber a chave de API como um header. Definimos a chave de API da aplicação no arquivo .env e já atualizamos a validação do .env para aceitá-la. Caso a chave recebida não bata com a chave da aplicação, retornamos um erro de status code 401 informando o motivo do erro. Caso contrário, a chave da API é retornada.

In [ ]:
from fastapi import Header, HTTPException, status
from app.config import API_SECRET_KEY

def validate_api_key(api_key: str = Header(..., alias="X-API-KEY", description="Chave da API")):
    if api_key != API_SECRET_KEY:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED, 
            detail="Acesso negado: Chave de API inválida ou ausente"
        )

    return api_key

Para utilizar a função e validar a chave de API recebida, temos dois métodos. No primeiro, recebemos a chave de API como argumento na função que processa a requisição e utilizamos a classe Depends, recebendo a função de validação como parâmetro. Já que o header na função não tinha valor padrão e portanto não era opcional, caso a requisição seja enviada para essa rota sem o header, será negada. Caso seja enviada e não bata com a chave autorizada, será negada.

In [ ]:
@app.get("/")
def root(api_key: str = Depends(validate_api_key)):
    return {"message": f"Acesso liberado com a chave: {api_key}"}

Mas, porque definir um argumento para a função que guarda a chave de API se não vamos utilizá-la dentro da rota? Para evitar ter uma variável desnecessária, também podemos utilizar o parâmetro dependencies do decorator que define o método da rota. Esse parâmetro receberá uma lista com todas as dependências que queremos injetar. Para aplicar a validação e verificação da chave de API, basta adicionarmos um objeto da classe Depends, que recebe como parâmetro a função que valida a chave.

In [ ]:
@app.get("/", dependencies=[Depends(validate_api_key)])
def root():
    return {"message": "Acesso liberado"}

Como de costume, o método simples tem suas desvantagens. Dentre elas, esse método não é integrado ao Swagger. Ou seja: a documentação não faz a mínima ideia de que rotas autorizadas precisam receber esse header. A outra é a vulnerabilidade a timing attacks. Comparações de strings com ```==``` ou ```!=``` retornam um resultado assim que encontram o primeiro caractere diferente. Um atacante medindo o tempo de requisição por nanossegundos conseguiria deduzir a chave secreta caractere por caractere até extrair o valor completo dela. Conforme o tempo de requisição aumenta, mais caracteres estão corretos.

### Método Robusto

Para transicionar da simplicidade para a robustez da autenticação, mudaremos a abordagem de forma incremental para que seja mais didático. Queremos resolver dois principais problemas: a falta de suporte na documentação e a vulnerabilidade a timing attacks. Primeiro, vamos trabalhar na documentação.

A primeira mudança é deixar de utilizar a classe Header diretamente para receber a chave de API. Ao invés disso, vamos definir um esquema que instancia a classe APIKeyHeader de fastapi.security. Essa classe pode receber um parâmetro name que informa o nome do header que tem de ser recebido. A classe APIKeyHeader é a responsável por adicionar o botão de autenticação à documentação. Para utilizar o header dessa forma, temos de injetar essa dependência na função utilizando Depends ou a classe Security. Tanto Depends como Security cumprem o mesmo papel, que é injetar uma dependência. A grande diferença entre os dois é que a classe Security possui parâmetros que a classe Depends não possui e que podem ser essenciais em alguns momentos especificamente para soluções de autenticação.

In [ ]:
from fastapi import Security
from fastapi import HTTPException, status
from fastapi.security import APIKeyHeader
from app.config import API_SECRET_KEY

api_key_scheme = APIKeyHeader(name="X-API-KEY")

def validate_api_key(api_key: str = Security(api_key_scheme)):
    if api_key != API_SECRET_KEY:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED, 
            detail="Acesso negado: Chave de API inválida ou ausente"
        )

    return api_key

Observe uma requisição enviada para uma rota de predição. Percebe que, quando não enviamos o header, recebemos de volta uma mensagem de erro genérica? Isso acontece porque o comportamento padrão é que quando o FastAPI percebe que a requisição não contém o header necessário, é lançado um erro automaticamente.

```shell 
curl -X POST http://localhost:8000/predict/single
```

**Resposta**

```json
{ 
    "detail": "Not authenticated" 
}
```

Para contornar esse problema, podemos passar o parâmetro auto_error para a classe APIKeyHeader. Esse argumento define se na falta do header o FastAPI deve ou não lançar um erro automaticamente. Como queremos uma mensagem customizada, desabilitamos o erro automático para lidarmos nós mesmos com a falta do header.

In [ ]:
from fastapi import Security
from fastapi import HTTPException, status
from fastapi.security import APIKeyHeader
from app.config import API_SECRET_KEY

api_key_scheme = APIKeyHeader(name="X-API-KEY", auto_error=False)

def validate_api_key(api_key: str = Security(api_key_scheme)):
    if api_key != API_SECRET_KEY:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED, 
            detail="Acesso negado: Chave de API inválida ou ausente"
        )

    return api_key

E essa é a nova resposta quando enviamos uma requisição.

```shell 
curl -X POST http://localhost:8000/predict/single
```

**Resposta**

```json
{ 
  "detail": "Acesso negado: Chave de API inválida ou ausente"
}
```

Agora que resolvemos o primeiro problema que era a documentação, podemos ir para o segundo problema: os timing attacks. Para resolver esse problema, temos de deixar de utilizar comparações com ```==``` ou ```!=```. Para isso, podemos substituir pela biblioteca criptográfica nativa "secrets". Ao comparar se a chave recebida bate com uma outra chave (Ex: a chave autorizada do .env), utilizamos o método secrets.compare_digest, que deverá receber a chave recebida e a chave autorizada, respectivamente. Agora, fomos inclusive mais específicos na mensagem de erro. Primeiro verificamos se a chave foi enviada, e se não foi, informamos que está ausente. Caso tenha sido enviada, verificamos se está correta. E se não está, informamos que a chave enviada é inválida. Dessa forma, um hacker mal intencionado não será mais capaz de deduzir a chave da API por timing attacks.

In [ ]:
import secrets
from fastapi import Security
from fastapi import HTTPException, status
from fastapi.security import APIKeyHeader
from app.config import API_SECRET_KEY

API_KEY_HEADER_NAME = "X-API-KEY"

api_key_scheme = APIKeyHeader(name=API_KEY_HEADER_NAME, auto_error=False)

def validate_api_key(api_key: str = Security(api_key_scheme)):
    if not api_key:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED, 
            detail=f"Acesso negado: Chave de API ('{API_KEY_HEADER_NAME}') ausente"
        )

    api_key_matches = secrets.compare_digest(api_key, API_SECRET_KEY)

    if not api_key_matches:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail=f"Acesso negado: Chave de API ('{API_KEY_HEADER_NAME}') inválida"
        )

    return api_key


### Gerando uma Chave de API

Idealmente, as chaves de API devem ter uma alta entropia de aleatoriedade. Então, ao invés de simplesmente pensar em uma chave, é mais seguro gerar uma utilizando a própria biblioteca nativa secrets. Conforme o comando abaixo que pode ser executado em um terminal.

```shell
python -c "import secrets; print(secrets.token_urlsafe(32))"
```

### Aplicando Autenticação para Roteadores

Atualmente, é assim que estamos aplicando a validação de chave de API nas rotas. Nessa abordagem, podemos utilizar a validação pontualmente, rota por rota.

In [ ]:
# ...

router = APIRouter(prefix="/predict", tags=["Predições"])

@router.post("/single", dependencies=[Security(validate_api_key)], response_model=ChurnOutput)
def predict_single(
    data: ChurnInput,
    ml_service: MLService = Depends(get_ml_service),
):
    return ml_service.predict_single(data)


@router.post("/batch", dependencies=[Security(validate_api_key)], response_model=list[ChurnOutput])
def predict_batch(
    data: list[ChurnInput],
    ml_service: MLService = Depends(get_ml_service),
):
    return ml_service.predict_batch(data)


Porém, se quisermos adicionar uma dependência (nesse caso a função de validação) para todas as rotas de um roteador, podemos passar o parâmetro dependencies incluindo a dependência desejada diretamente para a classe APIRouter.

In [ ]:
# ...

router = APIRouter(prefix="/predict", dependencies=[Security(validate_api_key)], tags=["Predições"])


@router.post("/single", response_model=ChurnOutput)
def predict_single(
    data: ChurnInput,
    ml_service: MLService = Depends(get_ml_service),
):
    return ml_service.predict_single(data)


@router.post("/batch", response_model=list[ChurnOutput])
def predict_batch(
    data: list[ChurnInput],
    ml_service: MLService = Depends(get_ml_service),
):
    return ml_service.predict_batch(data)


## Ciclo de Vida

Atualmente, na nossa aplicação, se tentarmos utilizar uma rota de predição, será lançado um erro informando "O modelo não foi carregado". Se lembra que é esse erro que é lançado quando tentamos fazer uma predição sem ter carregado um erro ainda lá na classe MLService? Isso é porque até o momento, não encontramos um lugar ideal para carregar o modelo. Não seria viável carregá-lo a cada predição, mas sim quando a API for executada. Além disso, é muito recomendável "jogar fora" o modelo quando a execução da API tiver de ser interrompida. Para isso, configuramos o ciclo de vida da aplicação.

Mas antes, temos de implementar na classe MLService um método que descarregue o modelo. Afinal, nesse momento ele tem apenas um método para carregar o modelo e os métodos de predição.

Então antes de tudo, implementamos o método unload_model na classe MLService. Esse método irá atribuir o valor None para a propriedade model da classe. Mas, isso não basta. A função load_model lá do predict.py que se refere ao código específico do modelo utiliza lru_cache. Portanto, mesmo esvaziando a variável, o modelo ainda estará armazenado em cache. Para resolver isso, temos de importar a função e chamar o método cache_clear dela. Mas isso apenas limpou a referência que estava dentro do lru_cache. Para liberar de fato a memória RAM para o sistema operacional, temos de importar a biblioteca nativa gc (Garbage Collector) e chamar o método collect.

In [ ]:
# ...

import gc as garbage_colector
from ga4_churn_predictor.predict import load_model

churn_output_list_adapter = TypeAdapter(list[ChurnOutput])


class MLService:
    # ...

    def unload_model(self):
        self.model = None

        load_model.cache_clear()

        garbage_colector.collect()

    # ...

ml_service = MLService()


def get_ml_service() -> MLService:
    return ml_service


Para configurar um ciclo de vida, podemos definir uma função assíncrona que geralmente chamamos de lifespan. Essa função deve utilizar o decorator asynccontextmanager da biblioteca nativa contextlib. Elá poderá receber uma variável app que é exatamente a aplicação em execução. 

A função será dividida em duas partes: antes do yield e depois do yield. O que acontece é muito simples. Quando a API é executada, tudo que está antes do yield é executado também. Então o yield pausa a função enquanto a aplicação está disponível para receber requisições. 

Finalmente, quando a API estiver sendo encerrada, tudo que está após o yield será executado. No código abaixo, a sequência fica muito clara. Ao executar a aplicação, o modelo é imediatamente carregado. Antes de interromper a execução, o modelo é descarregado. Agora que configuramos o ciclo de vida, basta passar essa função lifespan como parâmetro para a classe FastAPI que inicia a aplicação.

In [ ]:
# ...

from app.services.ml_service import ml_service
from contextlib import asynccontextmanager


@asynccontextmanager
async def lifespan(app: FastAPI):
    ml_service.load_model()

    yield
    
    ml_service.unload_model()
    

app = FastAPI(
    title=API_TITLE,
    version=API_VERSION,
    description=API_DESCRIPTION,
    lifespan=lifespan
)

app.include_router(predict_router)


## Middlewares

### Utilizando CORS

Um dos middlewares mais utilizados de toda a internet é o CORS, que controla o nível de acesso das origens de requisição. O FastAPI disponibiliza nativamente esse middleware. Basta importá-lo e depois configurar o middleware na aplicação. Para isso, chamamos o método add_middleware de app. Como primeiro parâmetro, passamos exatamente a classe do middleware. O restante dos argumentos é completamente livre, a depender do que aquele middleware específico precisa. Os principais ao utilizar CORS são: allow_origins; allow_headers; allow_methods; allow_credentials. Esses parâmetros definem de quais origens a aplicação deve aceitar requisições, quais headers deve aceitar, quais métodos de requisição deve aceitar e se a aplicação deve permitir o compartilhamento de credenciais como cookies de sessão e tokens de autenticação.

In [ ]:
# ...

from fastapi.middleware.cors import CORSMiddleware

# ...

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_headers=["*"],
    allow_methods=["*"],
    allow_credentials=True
)


### Middlewares Customizados

Nem sempre haverá um middleware pronto para tudo que precisarmos. Portanto, o FastAPI permite a definição de middlewares customizados. Para adicionar um middleware, definimos uma função assíncrona que recebe o objeto de requisição e uma função call_next. A função deverá utilizar o decorator app.middleware, passando como parâmetro para ele o tipo do middleware (nesse caso, "http"). Para lembrar, a função call_next libera o acesso para as rotas. O middleware abaixo calcula o tempo de processamento da API. 

Para isso, iniciamos um contador utilizando a biblioteca nativa time, esperamos a rota processar a requisição chamando a função assíncrona call_next e passando para ela o objeto de requisição. Quando a requisição for processada, a função call_next retornará o objeto de resposta. Nele, definimos um header chamado "X-Process-Time-Ms" que guarda a duração. Por fim, retornamos a resposta. Agora, todas as requisições da API retornarão para o cliente com um header que informa quanto tempo a API demorou para processar a requisição.

In [ ]:
# ...

import time
from fastapi import Request

@app.middleware("http")
async def add_process_time(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    duration_in_ms = (time.perf_counter() - start) * 1000
    response.headers["X-Process-Time-Ms"] = f"{duration_in_ms:.2f}"

    return response

## Tratamento de Exceções

### Lançando Exceções

Em qualquer aplicação, é crucial tratar as exceções a fim de fazer com que o sistema seja mais compreensível para o usuário. Além de ocultar informações que não convém que sejam reveladas como mensagens de erro muito específicas ou stack traces. Para isso, podemos tratar situações que poderiam causar erros de forma específica como no caso abaixo, em que verificamos se a chave da API foi enviada antes de validar.

Sem esse tratamento de erro customizado, se tentarmos enviar uma requisição para uma rota autorizada sem uma chave de API, o conteúdo retornado na resposta é "Internal Server Error". Completamente genérico, impossível de compreender qual foi o problema. 

O código é muito intuitivo, mas ainda vale a pena explicar. Para lançar uma exceção que retorne como resposta para o usuário, devemos utilizar a classe HTTPException. Ela poderá receber um parâmetro status code que representa é o código de erro HTTP e um parâmetro detail, que descreve a exceção. O FastAPI disponibiliza um objeto importado status, que guarda diversas constantes para códigos de erro HTTP. No exemplo abaixo, utilizamos status.HTTP_401_UNAUTHORIZED que é o código de erro para acesso não autorizado. Código 401.

In [ ]:
# ...

from fastapi import HTTPException, status

def validate_api_key(api_key: str = Security(api_key_scheme)):
    if not api_key:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED, 
            detail=f"Acesso negado: Chave de API ('{API_KEY_HEADER_NAME}') ausente"
        )

    api_key_matches = secrets.compare_digest(api_key, API_SECRET_KEY)

    if not api_key_matches:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail=f"Acesso negado: Chave de API ('{API_KEY_HEADER_NAME}') inválida"
        )

    return api_key


### Tratamento Global de Exceções

As exceções HTTP são as que lançamos quando há algum problema no processamento das requisições. Já as Runtime Exceptions são as que lançamos quando há algum erro que não se refere especificamente ao processamento das requisições. Para lançar exceções de runtime, utilizamos a classe RuntimeError nativa do python. Como no nosso MLService quando verificamos se a classe possui um modelo carregado.

In [ ]:
# ...

class MLService:
    # ...

    def predict_single(self, data: ChurnInput) -> ChurnOutput:
        if self.model is None:
            raise RuntimeError("O modelo não foi carregado")

        # ...


Também é necessário tratar estas exceções, assim como as que ocorrem durante o processo das requisições. As exceções HTTP geralmente facilitam compreender o problema que ocorreu ao processar uma requisição. Já as exceções de runtime geralmente têm como objetivo informar ao desenvolvedor o que aconteceu de errado durante a execução da aplicação. São essas as exceções que não devem chegar para o usuário. Para evitar que isso aconteça, podemos implementar uma função que lida com as exceções de forma global.

Para lidar com as exceções, definimos uma função que recebe um objeto de requisição e a exceção. Dentro dessa função, podemos retornar uma instância da classe JSONResponse do FastAPI. Essa classe monta uma resposta em formato JSON. Ela pode receber um parâmetro status_code que determina qual é o código de status que será retornado e um parâmetro content, que determina o conteúdo em JSON a ser retornado. Essa função tem de utilizar o decorator app.exception_handler, que recebe como parâmetro a classe RuntimeError, indicando que essa função de tratamento de erro deve tratar especificamente erros dessa classe.

In [ ]:
# ...

from fastapi.responses import JSONResponse
from fastapi import status

@app.exception_handler(RuntimeError)
def handle_runtime_error(request: Request, exception: RuntimeError):
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={"detail": "Erro interno do servidor"},
    )

O exemplo abaixo que utiliza a classe Exception ao invés de RuntimeError é ainda mais genérico. Já que a enorme maioria das exceções do python herdam de Exception, esse handler consegue capturar a enorme maioria de todas as exceções.

In [ ]:
# ...

from fastapi.responses import JSONResponse
from fastapi import status

@app.exception_handler(Exception)
def handle_exception(request: Request, exception: Exception):
    return JSONResponse(
        status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
        content={"detail": "Erro interno do servidor"},
    )

O decorator exception_handler também pode receber um código de status. Dessa forma, ele irá capturar os erros gerados automaticamente pelo FastAPI que tenham aquele código de erro. Por exemplo, quando tentamos enviar uma requisição para o FastAPI em uma rota que não existe, o FastAPI lança um erro automaticamente com o código de erro 404. O handler abaixo capturará esse erro e retonará uma resposta customizada para o cliente.

In [ ]:
# ...

from fastapi.responses import JSONResponse
from fastapi import status

@app.exception_handler(status.HTTP_404_NOT_FOUND)
def handle_not_found_exception(request: Request, exception: Exception):
    return JSONResponse(
        status_code=status.HTTP_404_NOT_FOUND,
        content={"detail": "Recurso não encontrado"},
    )

### Status Codes Mais Utilizados

| Nome da Propriedade | Tipo de Código | Status Code | Significado |
| ------------------- | -------------- | ----------- | ----------- |
| status.HTTP_200_OK | Sucesso | 200 | Processado sem erro |
| status.HTTP_201_CREATED | Sucesso | 201 | Recurso criado |
| status.HTTP_202_ACCEPTED | Sucesso | 202 | Requisição aceita |
| status.HTTP_204_NO_CONTENT | Sucesso | 204 | Resposta sem conteúdo |
| status.HTTP_400_BAD_REQUEST | Erro | 400 | Requisição inadequada |
| status.HTTP_401_UNAUTHORIZED | Erro | 401 | Acesso não autorizado |
| status.HTTP_402_PAYMENT_REQUIRED | Erro | 402 | Pagamento requerido |
| status.HTTP_403_FORBIDDEN | Erro | 403 | Acesso proibido |
| status.HTTP_404_NOT_FOUND | Erro | 404 | Recurso não encontrado |
| status.HTTP_409_CONFLICT | Erro | 409 | Conflito |
| status.HTTP_422_UNPROCESSABLE_CONTENT | Erro | 422 | Conteúdo não processável |
| status.HTTP_429_TOO_MANY_REQUESTS | Erro | 429 | Excesso de requisições |
| status.HTTP_500_INTERNAL_SERVER_ERROR | Erro | 500 | Erro interno do servidor |
| status.HTTP_501_NOT_IMPLEMENTED | Erro | 501 | Não implementado |
| status.HTTP_503_SERVICE_UNAVAILABLE | Erro | 503 | Serviço indisponível |